# 🔬 RetinaSense — Upgraded Pipeline
### EfficientNet-B3 | Ben Graham Preprocessing | Grad-CAM | Full Evaluation
---
**Upgrades over original:**
- ✅ EfficientNet-B3 backbone (was ResNet50)
- ✅ Ben Graham fundus preprocessing
- ✅ Stratified train/val split
- ✅ Auto class weights from data
- ✅ Cosine Annealing LR scheduler
- ✅ Progressive backbone unfreezing
- ✅ Gradient clipping
- ✅ Grad-CAM visualization
- ✅ Full evaluation: AUC, F1, Confusion Matrix, ROC
---
**Set EPOCHS = 4 for test run | Change to 50 for full training**

## STEP 0 — GPU Check

In [10]:
import torch
if torch.cuda.is_available():
    print('✅ GPU:', torch.cuda.get_device_name(0))
    print('✅ VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    print('❌ No GPU — Runtime > Change runtime type > T4 GPU')

✅ GPU: NVIDIA H200
✅ VRAM: 150.1 GB


## STEP 1 — Clone Repository

In [11]:
# !git clone https://github.com/Tanishq74/Retinal-Disease-MultiTask-AI.git
# %cd Retinal-Disease-MultiTask-AI
# print('✅ Repo cloned')

## STEP 2 — Install Dependencies

In [12]:
!pip install -q grad-cam scikit-learn opencv-python-headless seaborn
print('✅ Dependencies installed')

✅ Dependencies installed


## STEP 3 — Kaggle API Setup

In [13]:
# import os
# from google.colab import drive
# drive.mount('/content/drive')

# # Save outputs here
# SAVE_DIR = '/content/drive/MyDrive/RetinaSense'
# os.makedirs(SAVE_DIR, exist_ok=True)

# # Setup Kaggle — copies from Drive if available, else manual upload
# os.makedirs('/root/.kaggle', exist_ok=True)
# drive_kaggle = '/content/drive/MyDrive/kaggle.json'

# if os.path.exists(drive_kaggle):
#     !cp {drive_kaggle} /root/.kaggle/kaggle.json
#     !chmod 600 /root/.kaggle/kaggle.json
#     print('✅ kaggle.json loaded from Drive')
# elif os.path.exists('/root/.kaggle/kaggle.json'):
#     print('✅ kaggle.json already present')
# else:
#     from google.colab import files
#     print('⚠️  Upload your kaggle.json')
#     files.upload()
#     !mv kaggle.json /root/.kaggle/kaggle.json
#     !chmod 600 /root/.kaggle/kaggle.json
#     !cp /root/.kaggle/kaggle.json {drive_kaggle}
#     print('✅ kaggle.json saved to Drive for future use')

# !kaggle --version

## STEP 4 — Download Datasets

In [14]:
# import os

# # ── ODIR-5K ───────────────────────────────────────
# if not os.path.exists('/content/Retinal-Disease-MultiTask-AI/odir/full_df.csv'):
#     print('📥 Downloading ODIR-5K...')
#     !kaggle datasets download -d andrewmvd/ocular-disease-recognition-odir5k
#     !unzip -q ocular-disease-recognition-odir5k.zip -d odir
#     print('✅ ODIR ready')
# else:
#     print('✅ ODIR already present — skipping')

# # ── APTOS 2019 ────────────────────────────────────
# if not os.path.exists('/content/Retinal-Disease-MultiTask-AI/aptos/train.csv'):
#     print('📥 Downloading APTOS 2019 (~9GB, takes 5-10 mins)...')
#     !kaggle competitions download -c aptos2019-blindness-detection
#     !unzip -q aptos2019-blindness-detection.zip -d aptos
#     print('✅ APTOS ready')
# else:
#     print('✅ APTOS already present — skipping')

# # ── Verify ────────────────────────────────────────
# odir_imgs  = len(os.listdir('odir/preprocessed_images')) if os.path.exists('odir/preprocessed_images') else 0
# aptos_imgs = len(os.listdir('aptos/train_images'))       if os.path.exists('aptos/train_images')       else 0
# print(f'\nODIR  images : {odir_imgs}')
# print(f'APTOS images : {aptos_imgs}')

## STEP 5 — All Imports

In [15]:
import os, time, warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASS_NAMES = ['Normal', 'Diabetes/DR', 'Glaucoma', 'Cataract', 'AMD']
print(f'✅ All imports done | Device: {device}')

✅ All imports done | Device: cuda


## STEP 6 — Build Unified Metadata
> Same logic as original but with auto-detection of ODIR columns

In [16]:
BASE = './'
ODIR_CSV      = f'{BASE}/odir/full_df.csv'
ODIR_IMG_DIR  = f'{BASE}/odir/preprocessed_images'
APTOS_CSV     = f'{BASE}/aptos/train.csv'
APTOS_IMG_DIR = f'{BASE}/aptos/train_images'

disease_cols = ['N', 'D', 'G', 'C', 'A']
label_map    = {'N': 0, 'D': 1, 'G': 2, 'C': 3, 'A': 4}

# ── ODIR Metadata (same logic as original) ─────────
df_odir = pd.read_csv(ODIR_CSV)
print('ODIR columns:', list(df_odir.columns))

df_odir['disease_count'] = df_odir[disease_cols].sum(axis=1)
df_odir = df_odir[df_odir['disease_count'] == 1].copy()

def get_label(row):
    for d in disease_cols:
        if row[d] == 1:
            return label_map[d]

df_odir['disease_label'] = df_odir.apply(get_label, axis=1)

# Auto-detect image filename column
img_col = None
for col in df_odir.columns:
    if 'filename' in col.lower() or 'fundus' in col.lower() or 'image' in col.lower():
        img_col = col
        break
print(f'ODIR image column: {img_col}')

odir_metadata = pd.DataFrame({
    'image_path'    : ODIR_IMG_DIR + '/' + df_odir[img_col].astype(str),
    'dataset'       : 'ODIR',
    'disease_label' : df_odir['disease_label'],
    'severity_label': -1
})
print(f'ODIR samples: {len(odir_metadata)}')

# ── APTOS Metadata ─────────────────────────────────
df_aptos = pd.read_csv(APTOS_CSV)
aptos_metadata = pd.DataFrame({
    'image_path'    : APTOS_IMG_DIR + '/' + df_aptos['id_code'] + '.png',
    'dataset'       : 'APTOS',
    'disease_label' : 1,
    'severity_label': df_aptos['diagnosis']
})
print(f'APTOS samples: {len(aptos_metadata)}')

# ── Merge & Clean ──────────────────────────────────
final_metadata = pd.concat([odir_metadata, aptos_metadata], ignore_index=True)
print(f'Before path check: {len(final_metadata)}')
final_metadata = final_metadata[
    final_metadata['image_path'].apply(os.path.exists)
].reset_index(drop=True)
print(f'After path check : {len(final_metadata)}')

final_metadata.to_csv('final_unified_metadata.csv', index=False)
final_metadata.to_csv(f'{SAVE_DIR}/metadata.csv', index=False)

print('\n📊 Class distribution:')
for i, cnt in final_metadata['disease_label'].value_counts().sort_index().items():
    print(f'   {CLASS_NAMES[i]:15s}: {cnt}')

ODIR columns: ['ID', 'Patient Age', 'Patient Sex', 'Left-Fundus', 'Right-Fundus', 'Left-Diagnostic Keywords', 'Right-Diagnostic Keywords', 'N', 'D', 'G', 'C', 'A', 'H', 'M', 'O', 'filepath', 'labels', 'target', 'filename']
ODIR image column: Left-Fundus
ODIR samples: 4966
APTOS samples: 3662
Before path check: 8628
After path check : 8540


NameError: name 'SAVE_DIR' is not defined

## STEP 7 — Ben Graham Preprocessing + Dataset Class
> **Upgrade:** Ben Graham local contrast normalization for fundus images

In [ ]:
def ben_graham_preprocess(img_path, img_size=224, sigmaX=10):
    """Ben Graham fundus preprocessing — enhances retinal structures."""
    img = cv2.imread(img_path)
    if img is None:
        img = np.array(Image.open(img_path).convert('RGB'))
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (img_size, img_size))
    img = cv2.addWeighted(img, 4,
                          cv2.GaussianBlur(img, (0,0), sigmaX), -4, 128)
    mask = np.zeros(img.shape[:2], dtype=np.uint8)
    cv2.circle(mask, (img_size//2, img_size//2), int(img_size*0.48), 255, -1)
    img = cv2.bitwise_and(img, img, mask=mask)
    return Image.fromarray(img)


def get_transforms(phase='train'):
    if phase == 'train':
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.2),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])
    else:
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])


class RetinalDataset(Dataset):
    """Same interface as original but with Ben Graham preprocessing."""
    def __init__(self, dataframe, transform=None, use_ben_graham=True):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.use_ben_graham = use_ben_graham

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            image = ben_graham_preprocess(row['image_path']) if self.use_ben_graham \
                    else Image.open(row['image_path']).convert('RGB')
        except Exception:
            image = Image.new('RGB', (224, 224), (128, 128, 128))
        if self.transform:
            image = self.transform(image)
        return (
            image,
            torch.tensor(int(row['disease_label']),  dtype=torch.long),
            torch.tensor(int(row['severity_label']), dtype=torch.long)
        )


# Visualise preprocessing
sample = final_metadata[final_metadata['dataset']=='APTOS'].iloc[0]
orig   = Image.open(sample['image_path']).convert('RGB').resize((224,224))
proc   = ben_graham_preprocess(sample['image_path'])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(orig);  axes[0].set_title('Original Fundus',         fontsize=13); axes[0].axis('off')
axes[1].imshow(proc);  axes[1].set_title('Ben Graham Preprocessed', fontsize=13); axes[1].axis('off')
plt.suptitle('Preprocessing Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/preprocessing.png', dpi=150)
plt.show()
print('✅ Dataset class and preprocessing ready')

## STEP 8 — DataLoaders
> **Upgrade:** Stratified split (was random_split)

In [ ]:
BATCH_SIZE = 32

# Stratified split — ensures equal class ratio in train and val
train_df, val_df = train_test_split(
    final_metadata,
    test_size=0.2,
    stratify=final_metadata['disease_label'],
    random_state=42
)

train_dataset = RetinalDataset(train_df, transform=get_transforms('train'), use_ben_graham=True)
val_dataset   = RetinalDataset(val_df,   transform=get_transforms('val'),   use_ben_graham=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'✅ Train: {len(train_dataset)} samples ({len(train_loader)} batches)')
print(f'   Val  : {len(val_dataset)}  samples ({len(val_loader)}  batches)')

# Class distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = ['#2ecc71','#3498db','#e74c3c','#f39c12','#9b59b6']
for ax, df, title in zip(axes, [train_df, val_df], ['Train', 'Validation']):
    counts = df['disease_label'].value_counts().sort_index()
    bars = ax.bar([CLASS_NAMES[i] for i in counts.index], counts.values, color=colors)
    ax.set_title(f'{title} Set Distribution', fontsize=13, fontweight='bold')
    ax.set_ylabel('Count')
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x()+bar.get_width()/2, v+5, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/class_distribution.png', dpi=150)
plt.show()

## STEP 9 — EfficientNet-B3 Multi-Task Model
> **Upgrade:** EfficientNet-B3 backbone + deeper heads with BatchNorm (was ResNet50 + single Linear)

In [ ]:
class MultiTaskModel(nn.Module):
    def __init__(self, num_disease_classes=5, num_severity_classes=5, dropout=0.4):
        super().__init__()

        # ── EfficientNet-B3 backbone ───────────────
        backbone = models.efficientnet_b3(weights='IMAGENET1K_V1')
        self.backbone    = nn.Sequential(*list(backbone.children())[:-1])
        self.feature_dim = 1536  # EfficientNet-B3 output
        self.dropout     = nn.Dropout(dropout)

        # ── Disease classification head ────────────
        self.disease_head = nn.Sequential(
            nn.Linear(self.feature_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_disease_classes)
        )

        # ── Severity grading head ──────────────────
        self.severity_head = nn.Sequential(
            nn.Linear(self.feature_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_severity_classes)
        )

    def forward(self, x):
        f = self.backbone(x)
        f = f.view(f.size(0), -1)
        f = self.dropout(f)
        return self.disease_head(f), self.severity_head(f)


model = MultiTaskModel().to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('✅ EfficientNet-B3 Multi-Task Model')
print(f'   Total params    : {total:,}')
print(f'   Trainable params: {trainable:,}')

## STEP 10 — Loss, Optimizer & Scheduler
> **Upgrades:** Auto class weights | Cosine Annealing scheduler | Progressive unfreezing

In [ ]:
# ── Auto class weights from actual data ────────────
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0,1,2,3,4]),
    y=train_df['disease_label'].values
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print('Class weights:')
for name, w in zip(CLASS_NAMES, class_weights):
    print(f'  {name:15s}: {w:.3f}')

# ── Loss functions ─────────────────────────────────
criterion_disease  = nn.CrossEntropyLoss(weight=class_weights_tensor)
criterion_severity = nn.CrossEntropyLoss(ignore_index=-1)  # -1 = ODIR (no severity)

# ── Phase 1: freeze backbone, train heads only ─────
for param in model.backbone.parameters():
    param.requires_grad = False

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-7
)
scaler = GradScaler()

# ── CONFIG — change EPOCHS=50 for full training ────
EPOCHS        = 4   # ← 4 for test | 50 for full training
WARMUP_EPOCHS = 2   # Heads-only before full unfreeze
CHECKPOINT    = f'{SAVE_DIR}/RetinaSense_best_model.pth'

print(f'\n✅ Training config ready')
print(f'   Epochs        : {EPOCHS}  (change to 50 for full run)')
print(f'   Warmup epochs : {WARMUP_EPOCHS} (heads only)')
print(f'   Batch size    : {BATCH_SIZE}')
print(f'   Checkpoint    : {CHECKPOINT}')

## STEP 11 — Training Loop
> **Upgrades:** Progressive unfreezing | Gradient clipping | Cosine LR | Batch logging

In [ ]:
history  = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best_acc = 0.0

for epoch in range(EPOCHS):
    start = time.time()

    # ── Unfreeze backbone after warmup ──────────────
    if epoch == WARMUP_EPOCHS:
        print('\n🔓 Unfreezing full backbone')
        for param in model.backbone.parameters():
            param.requires_grad = True
        optimizer = torch.optim.Adam(
            model.parameters(), lr=5e-5, weight_decay=1e-4
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=10, T_mult=2, eta_min=1e-7
        )

    # ── Train ────────────────────────────────────────
    model.train()
    total_loss = 0.0
    train_correct = 0
    train_total   = 0

    for batch_idx, (images, disease_labels, severity_labels) in enumerate(train_loader):
        images         = images.to(device)
        disease_labels = disease_labels.to(device)
        severity_labels= severity_labels.to(device)

        optimizer.zero_grad()

        with autocast('cuda'):
            disease_out, severity_out = model(images)
            loss_d = criterion_disease(disease_out, disease_labels)
            loss_s = criterion_severity(severity_out, severity_labels)
            loss   = loss_d + 0.5 * loss_s

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clip
        scaler.step(optimizer)
        scaler.update()

        total_loss    += loss.item()
        preds          = disease_out.argmax(dim=1)
        train_correct += (preds == disease_labels).sum().item()
        train_total   += disease_labels.size(0)

        if batch_idx % 20 == 0:
            print(f'  Epoch {epoch+1}/{EPOCHS} | '
                  f'Batch {batch_idx}/{len(train_loader)} | '
                  f'Loss: {loss.item():.4f}', end='\r')

    train_acc  = 100 * train_correct / train_total
    avg_loss   = total_loss / len(train_loader)

    # ── Validation ───────────────────────────────────
    model.eval()
    val_correct = 0
    val_total   = 0
    val_loss    = 0.0

    with torch.no_grad():
        for images, disease_labels, severity_labels in val_loader:
            images          = images.to(device)
            disease_labels  = disease_labels.to(device)
            severity_labels = severity_labels.to(device)
            with autocast('cuda'):
                disease_out, severity_out = model(images)
                loss_d = criterion_disease(disease_out, disease_labels)
                loss_s = criterion_severity(severity_out, severity_labels)
                loss   = loss_d + 0.5 * loss_s
            val_loss    += loss.item()
            preds        = disease_out.argmax(dim=1)
            val_correct += (preds == disease_labels).sum().item()
            val_total   += disease_labels.size(0)

    val_acc  = 100 * val_correct / val_total
    val_loss /= len(val_loader)
    scheduler.step()

    history['train_loss'].append(avg_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    lr = optimizer.param_groups[0]['lr']

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            'epoch'              : epoch,
            'model_state_dict'   : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc'            : val_acc,
            'history'            : history
        }, CHECKPOINT)
        print(f'\n🔥 Best model saved!')

    print(f'\n{"-"*50}')
    print(f'Epoch [{epoch+1}/{EPOCHS}] | LR: {lr:.2e}')
    print(f'Train Loss: {avg_loss:.4f} | Train Acc: {train_acc:.2f}%')
    print(f'Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.2f}%')
    print(f'Time: {round(time.time()-start, 2)} sec')
    print(f'{"-"*50}')

print(f'\n🏁 Training complete! Best Val Accuracy: {best_acc:.2f}%')

## STEP 12 — Training Curves

In [ ]:
epochs_range = range(1, len(history['train_loss'])+1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss')
axes[0].plot(epochs_range, history['val_loss'],   'r-o', label='Val Loss')
axes[0].set_title('Loss Curve', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history['train_acc'], 'b-o', label='Train Acc')
axes[1].plot(epochs_range, history['val_acc'],   'r-o', label='Val Acc')
axes[1].set_title('Accuracy Curve', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('RetinaSense — EfficientNet-B3 Training', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_curves.png', dpi=150)
plt.show()
print('✅ Training curves saved')

## STEP 13 — Full Evaluation (AUC, F1, Classification Report)

In [ ]:
# Load best model
ckpt = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'✅ Loaded best model — epoch {ckpt["epoch"]+1} | Val Acc: {ckpt["val_acc"]:.2f}%')

all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, disease_labels, _ in tqdm(val_loader, desc='Evaluating'):
        images = images.to(device)
        d_out, _ = model(images)
        probs = torch.softmax(d_out, dim=1)
        all_preds.extend(d_out.argmax(1).cpu().numpy())
        all_labels.extend(disease_labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

print('\n📊 Classification Report')
print('='*60)
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))

macro_f1    = f1_score(all_labels, all_preds, average='macro')
weighted_f1 = f1_score(all_labels, all_preds, average='weighted')
print(f'Macro F1    : {macro_f1:.4f}')
print(f'Weighted F1 : {weighted_f1:.4f}')

try:
    macro_auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')
    print(f'Macro AUC-ROC : {macro_auc:.4f}')
except Exception as e:
    macro_auc = 'N/A'
    print(f'AUC skipped: {e}')

pd.DataFrame([{
    'val_accuracy': ckpt['val_acc'],
    'macro_f1'    : macro_f1,
    'weighted_f1' : weighted_f1,
    'macro_auc'   : macro_auc
}]).to_csv(f'{SAVE_DIR}/metrics_summary.csv', index=False)
print('\n✅ Metrics saved to Drive')

## STEP 14 — Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm,      annot=True, fmt='d',   cmap='Blues',  ax=axes[0],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, linewidths=0.5)
axes[0].set_title('Confusion Matrix (Counts)',     fontsize=13, fontweight='bold')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')
axes[0].tick_params(axis='x', rotation=30)

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens', ax=axes[1],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, vmin=0, vmax=1)
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('RetinaSense — Disease Classification Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix saved')

## STEP 15 — ROC Curves (Per Class)

In [ ]:
y_bin  = label_binarize(all_labels, classes=[0,1,2,3,4])
colors = ['#2ecc71','#3498db','#e74c3c','#f39c12','#9b59b6']

plt.figure(figsize=(9, 7))
for i, (name, color) in enumerate(zip(CLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f'{name} (AUC = {auc(fpr, tpr):.3f})')

plt.plot([0,1],[0,1],'k--', lw=1.5, label='Random')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate',  fontsize=12)
plt.title('ROC Curves — Per Disease Class', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/roc_curves.png', dpi=150)
plt.show()
print('✅ ROC curves saved')

## STEP 16 — Grad-CAM Visualization

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Wrapper so Grad-CAM sees only disease head output
class DiseaseWrapper(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return self.m(x)[0]

wrapped      = DiseaseWrapper(model)
target_layer = [model.backbone[-1][0]]
cam          = GradCAM(model=wrapped, target_layers=target_layer)

inv_norm = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

def get_gradcam(img_path):
    tensor = get_transforms('val')(ben_graham_preprocess(img_path)).unsqueeze(0).to(device)
    with torch.no_grad():
        d_out, _ = model(tensor)
    pred = d_out.argmax(1).item()
    prob = torch.softmax(d_out, dim=1)[0][pred].item()
    gray = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(pred)])[0]
    img_d = np.clip(
        inv_norm(tensor.squeeze(0).cpu()).permute(1,2,0).numpy(), 0, 1
    ).astype(np.float32)
    return img_d, show_cam_on_image(img_d, gray, use_rgb=True), pred, prob


fig, axes = plt.subplots(5, 3, figsize=(12, 20))

for cls_id, cls_name in enumerate(CLASS_NAMES):
    samples = val_df[val_df['disease_label'] == cls_id]
    if len(samples) == 0: continue
    row = samples.sample(1, random_state=42).iloc[0]

    orig                       = ben_graham_preprocess(row['image_path'])
    img_d, cam_img, pred, prob = get_gradcam(row['image_path'])
    match = '✅' if pred == cls_id else '❌'

    axes[cls_id, 0].imshow(orig);    axes[cls_id, 0].axis('off')
    axes[cls_id, 0].set_title(f'Original\nTrue: {cls_name}', fontsize=10)

    axes[cls_id, 1].imshow(img_d);   axes[cls_id, 1].axis('off')
    axes[cls_id, 1].set_title('Ben Graham', fontsize=10)

    axes[cls_id, 2].imshow(cam_img); axes[cls_id, 2].axis('off')
    axes[cls_id, 2].set_title(
        f'Grad-CAM {match}\nPred: {CLASS_NAMES[pred]} ({prob:.2f})', fontsize=10
    )

plt.suptitle('RetinaSense — Grad-CAM Disease Localization',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/gradcam_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Grad-CAM saved')

## STEP 17 — Save Everything to Drive

In [ ]:
# Model already saved during training — just confirm
print('='*55)
print('        RETINASENSE — RESULTS SUMMARY')
print('='*55)
print(f'  Backbone      : EfficientNet-B3')
print(f'  Total samples : {len(final_metadata)}')
print(f'  Best Val Acc  : {ckpt["val_acc"]:.2f}%')
print(f'  Macro F1      : {macro_f1:.4f}')
try:
    print(f'  Macro AUC     : {macro_auc:.4f}')
except: pass
print('='*55)
print(f'\n📁 All outputs saved to Drive: {SAVE_DIR}')
print('   ├── RetinaSense_best_model.pth')
print('   ├── metadata.csv')
print('   ├── preprocessing.png')
print('   ├── class_distribution.png')
print('   ├── training_curves.png')
print('   ├── confusion_matrix.png')
print('   ├── roc_curves.png')
print('   ├── gradcam_visualization.png')
print('   └── metrics_summary.csv')
print('\n🎯 For full training: change EPOCHS=50 in STEP 10')